Nazario dataset exploration, structures the emails into a format that the ML models can use

Run on python 3.13.13 kernel, on Visual studios code, using juypter notebook
Generates "2_Naz_ml_ready.csv" which contains a label column and a text column, comprising of email subject + email body.
runs on the provided Nazario.csv file

In [2]:
pip install pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [4]:
#reading the dataset and viewing the first few rows
df = pd.read_csv("Nazario.csv")
print(df.head())
print(df.columns)
print(df.shape)

                                              sender  \
0  Mail System Internal Data <MAILER-DAEMON@monke...   
1                        cPanel <service@cpanel.com>   
2    Microsoft Outlook <recepcao@unimedceara.com.br>   
3                     Ann Garcia <AnGarcia@mcoe.org>   
4                 "USAA" <usaaacctupdate@sccu4u.com>   

                                 receiver  \
0                                     NaN   
1                         jose@monkey.org   
2                                     NaN   
3     "info@maaaaa.org" <info@maaaaa.org>   
4  Recipients <usaaacctupdate@sccu4u.com>   

                                    date  \
0             28 Sep 2017 09:57:25 -0400   
1        Fri, 30 Oct 2015 00:00:48 -0500   
2  Fri, 30 Oct 2015 06:21:59 -0300 (BRT)   
3        Fri, 30 Oct 2015 14:54:33 +0000   
4        Fri, 30 Oct 2015 14:02:33 -0500   

                                             subject  \
0  DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL DATA   
1              

In [5]:
import pandas as pd

# Keep only the columns needed
df = df[["subject", "body", "label"]]
#merging the subject and body
df["text"] = df["subject"].fillna("") + " " + df["body"].fillna("")
#keeping the only 2 columns needed
df = df[["text", "label"]]
#changing the label default text
df["label"] = "phishing"

print(df.head())
print(df.columns)

                                                text     label
0  DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL D...  phishing
1  Verify Your Account Business with  \t\t\t\t\t\...  phishing
2  Helpdesk Mailbox Alert!!! Your two incoming ma...  phishing
3  IT-Service Help Desk Password will expire in 3...  phishing
4  Final USAA Reminder - Update Your Account Now ...  phishing
Index(['text', 'label'], dtype='object')


In [6]:
import quopri
import re
print(df.head())

#cleaning text and removing inherited encoding issues
def clean_text(text):
    if pd.isna(text):
        return text
    
    # setting encoding to utf-8 encoding
    text = text.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
    
    # Decode quoted-printable
    try:
        text = quopri.decodestring(text).decode("utf-8", errors="ignore")
    except:
        pass
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

#applying clean
df["text"] = df["text"].apply(clean_text)
    
print(df.head())

from email.header import decode_header

import re
#still featuring encoding problems 
def remove_remaining_mime(text):
    if pd.isna(text):
        return text
    
    # Remove any leftover MIME patterns
    text = re.sub(r"=\?utf-8\?Q\?.*?\?=", "", text, flags=re.IGNORECASE)
    
    # Clean leftover fragments
    text = re.sub(r"=\?utf-8\?Q\?", "", text, flags=re.IGNORECASE)
    
    return text.strip()
#applying fix
df["text"] = df["text"].apply(remove_remaining_mime)
print("Check for remaining encoding errors")

mask = df["text"].str.contains(r"\=\?utf-8\?Q\?", na=False)
#checking number of rows remaining with encoding errors
print("Remaining encoded rows:", mask.sum())
df.to_csv("2_Naz_ml_ready.csv", index=False)



                                                text     label
0  DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL D...  phishing
1  Verify Your Account Business with  \t\t\t\t\t\...  phishing
2  Helpdesk Mailbox Alert!!! Your two incoming ma...  phishing
3  IT-Service Help Desk Password will expire in 3...  phishing
4  Final USAA Reminder - Update Your Account Now ...  phishing
                                                text     label
0  DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL D...  phishing
1  Verify Your Account Business with cPanel & WHM...  phishing
2  Helpdesk Mailbox Alert!!! Your two incoming ma...  phishing
3  IT-Service Help Desk Password will expire in 3...  phishing
4  Final USAA Reminder - Update Your Account Now ...  phishing
Check for remaining encoding errors
Remaining encoded rows: 0
